In [1]:
import pybamm
import numpy as np
import pandas as pd
import os

# ==========================
# 設定
# ==========================
N_SIM = 1000
TOTAL_TIME = 300
SAVE_DIR = "dataset_lstm"
os.makedirs(SAVE_DIR, exist_ok=True)

options = {
    "SEI": "ec reaction limited",
    "lithium plating": "irreversible",
}

for sim_id in range(N_SIM):

    print(f"Generating battery {sim_id}")

    k_sei = 10**np.random.uniform(-16, -13)
    k_plating = 10**np.random.uniform(-11, -8)
    dead_li = 10**np.random.uniform(-7, -4)

    eps_n = np.random.uniform(0.5, 0.7)
    eps_p = np.random.uniform(0.4, 0.6)

    r_n = np.random.uniform(5e-6, 15e-6)
    r_p = np.random.uniform(5e-6, 15e-6)

    T = np.random.uniform(273, 313)

    base_param = pybamm.ParameterValues("OKane2022")
    base_param.update({
        "SEI kinetic rate constant [m.s-1]": k_sei,
        "Lithium plating kinetic rate constant [m.s-1]": k_plating,
        "Lithium plating transfer coefficient": 0.5,
        "Dead lithium decay constant [s-1]": dead_li,
        "Negative electrode active material volume fraction": eps_n,
        "Positive electrode active material volume fraction": eps_p,
        "Negative particle radius [m]": r_n,
        "Positive particle radius [m]": r_p,
        "Ambient temperature [K]": T,
        "Initial temperature [K]": T,
    })

    v_init = np.random.uniform(3.3, 4.0)

    model_init = pybamm.lithium_ion.DFN(options=options)
    init_exp = pybamm.Experiment([
        f"Discharge at 0.2C until {v_init} V",
        "Rest for 2 hour"
    ])

    sim_init = pybamm.Simulation(
        model_init,
        parameter_values=base_param.copy(),
        experiment=init_exp
    )

    solution_init = sim_init.solve()

    drive_param = base_param.copy()
    drive_param.update({
        "Current function [A]": pybamm.InputParameter("Current function [A]")
    })

    model = pybamm.lithium_ion.DFN(options=options)
    model.set_initial_conditions_from(solution_init)

    sim = pybamm.Simulation(model, parameter_values=drive_param)
    sim.solve([0, 1], inputs={"Current function [A]": 0})

    esoh_solver = pybamm.lithium_ion.ElectrodeSOHSolver(
        parameter_values=base_param.copy()
    )

    time_log = []
    fcc_log = []
    current_log = []
    voltage_log = []

    I_now = 0
    I_max = np.random.uniform(1, 4)
    max_step = np.random.uniform(0.01, 0.2)

    valid_battery = True   # ← 追加（生存フラグ）

    for t in range(TOTAL_TIME):

        I_now += np.random.uniform(-max_step, max_step)
        I_now = np.clip(I_now, -I_max, I_max)

        sim.step(dt=1, inputs={"Current function [A]": I_now})
        solution = sim.solution

        Q_n = solution["Negative electrode capacity [A.h]"].entries[-1]
        Q_p = solution["Positive electrode capacity [A.h]"].entries[-1]
        Q_Li = solution["Total lithium capacity [A.h]"].entries[-1]

        try:
            esoh_sol = esoh_solver.solve({
                "Q_n": Q_n,
                "Q_p": Q_p,
                "Q_Li": Q_Li
            })
            FCC = esoh_sol["Q"]

        except pybamm.SolverError:
            print(f"⚠️ Battery {sim_id} skipped at t={t} (eSOH fail)")
            valid_battery = False
            break

        V = solution["Terminal voltage [V]"].entries[-1]

        time_log.append(t)
        fcc_log.append(FCC)
        current_log.append(-I_now)
        voltage_log.append(V)

    # ==========================
    # 保存（成功したものだけ）
    # ==========================
    if valid_battery and len(time_log) == TOTAL_TIME:

        df = pd.DataFrame({
            "time_s": time_log,
            "current_A": current_log,
            "voltage_V": voltage_log,
            "FCC_Ah": fcc_log
        })

        df.to_csv(f"{SAVE_DIR}/battery_{sim_id:04d}.csv", index=False)

    else:
        print(f"❌ Battery {sim_id} removed")

print("All datasets generated successfully.")

Generating battery 0
Generating battery 1
Generating battery 2
Generating battery 3
Generating battery 4
Generating battery 5
Generating battery 6
Generating battery 7
Generating battery 8
Generating battery 9
Generating battery 10
Generating battery 11
Generating battery 12
Generating battery 13
Generating battery 14
Generating battery 15
Generating battery 16
Generating battery 17


SolverError: Events ['Maximum voltage [V]'] are non-positive at initial conditions

In [2]:
import pybamm
import numpy as np
import pandas as pd
import os

# ==========================
# 設定
# ==========================
N_SIM = 1000
TOTAL_TIME = 300
SAVE_DIR = "dataset_lstm"
os.makedirs(SAVE_DIR, exist_ok=True)

options = {
    "SEI": "ec reaction limited",
    "lithium plating": "irreversible",
}

# ==========================
# 成功数管理
# ==========================
save_id = 0
trial_id = 0

# ==========================
# 成功するまで回す
# ==========================
while save_id < N_SIM:

    print(f"Generating battery {save_id} (trial {trial_id})")
    trial_id += 1

    # ==========================
    # 🎲 ランダム劣化
    # ==========================
    k_sei = 10**np.random.uniform(-16, -13)
    k_plating = 10**np.random.uniform(-11, -8)
    dead_li = 10**np.random.uniform(-7, -4)

    eps_n = np.random.uniform(0.5, 0.7)
    eps_p = np.random.uniform(0.4, 0.6)

    r_n = np.random.uniform(5e-6, 15e-6)
    r_p = np.random.uniform(5e-6, 15e-6)

    T = np.random.uniform(273, 313)

    base_param = pybamm.ParameterValues("OKane2022")
    base_param.update({
        "SEI kinetic rate constant [m.s-1]": k_sei,
        "Lithium plating kinetic rate constant [m.s-1]": k_plating,
        "Lithium plating transfer coefficient": 0.5,
        "Dead lithium decay constant [s-1]": dead_li,
        "Negative electrode active material volume fraction": eps_n,
        "Positive electrode active material volume fraction": eps_p,
        "Negative particle radius [m]": r_n,
        "Positive particle radius [m]": r_p,
        "Ambient temperature [K]": T,
        "Initial temperature [K]": T,
    })

    # ==========================
    # 🎲 初期SOCランダム化
    # ==========================
    v_init = np.random.uniform(3.3, 4.0)

    model_init = pybamm.lithium_ion.DFN(options=options)

    init_exp = pybamm.Experiment([
        f"Discharge at 0.2C until {v_init} V",
        "Rest for 2 hour"
    ])

    sim_init = pybamm.Simulation(
        model_init,
        parameter_values=base_param.copy(),
        experiment=init_exp
    )

    try:
        solution_init = sim_init.solve()
    except pybamm.SolverError:
        print("❌ init fail")
        continue

    # ==========================
    # Driveモデル
    # ==========================
    drive_param = base_param.copy()
    drive_param.update({
        "Current function [A]": pybamm.InputParameter("Current function [A]")
    })

    model = pybamm.lithium_ion.DFN(options=options)
    model.set_initial_conditions_from(solution_init)

    sim = pybamm.Simulation(model, parameter_values=drive_param)

    try:
        sim.solve([0, 1], inputs={"Current function [A]": 0})
    except pybamm.SolverError:
        print("❌ drive init fail")
        continue

    esoh_solver = pybamm.lithium_ion.ElectrodeSOHSolver(
        parameter_values=base_param.copy()
    )

    # ==========================
    # ログ
    # ==========================
    time_log = []
    fcc_log = []
    current_log = []
    voltage_log = []

    I_now = 0
    I_max = np.random.uniform(1, 4)
    max_step = np.random.uniform(0.01, 0.2)

    valid_battery = True

    # ==========================
    # 毎秒ループ
    # ==========================
    for t in range(TOTAL_TIME):

        solution = sim.solution
        V_now = solution["Terminal voltage [V]"].entries[-1]

        # 電圧制約
        if V_now > 4.19:
            I_now = min(I_now, 0)

        if V_now < 3.01:
            I_now = max(I_now, 0)

        # ランダム更新
        I_now += np.random.uniform(-max_step, max_step)
        I_now = np.clip(I_now, -I_max, I_max)

        # DFN step
        try:
            sim.step(dt=1, inputs={"Current function [A]": I_now})
        except pybamm.SolverError:
            print(f"⚠️ DFN fail at t={t}")
            valid_battery = False
            break

        solution = sim.solution

        Q_n = solution["Negative electrode capacity [A.h]"].entries[-1]
        Q_p = solution["Positive electrode capacity [A.h]"].entries[-1]
        Q_Li = solution["Total lithium capacity [A.h]"].entries[-1]

        # eSOH
        try:
            esoh_sol = esoh_solver.solve({
                "Q_n": Q_n,
                "Q_p": Q_p,
                "Q_Li": Q_Li
            })
            FCC = esoh_sol["Q"]
        except pybamm.SolverError:
            print(f"⚠️ eSOH fail at t={t}")
            valid_battery = False
            break

        V = solution["Terminal voltage [V]"].entries[-1]

        time_log.append(t)
        fcc_log.append(FCC)
        current_log.append(-I_now)
        voltage_log.append(V)

    # ==========================
    # 保存（成功セルのみ）
    # ==========================
    if valid_battery and len(time_log) == TOTAL_TIME:

        df = pd.DataFrame({
            "time_s": time_log,
            "current_A": current_log,
            "voltage_V": voltage_log,
            "FCC_Ah": fcc_log
        })

        df.to_csv(f"{SAVE_DIR}/battery_{save_id:04d}.csv", index=False)
        save_id += 1

    else:
        print("❌ battery removed")

print("All datasets generated successfully.")

Generating battery 0 (trial 0)
Generating battery 1 (trial 1)
Generating battery 2 (trial 2)
Generating battery 3 (trial 3)
Generating battery 4 (trial 4)
Generating battery 5 (trial 5)
Generating battery 6 (trial 6)
Generating battery 7 (trial 7)
Generating battery 8 (trial 8)
Generating battery 9 (trial 9)
Generating battery 10 (trial 10)
Generating battery 11 (trial 11)
Generating battery 12 (trial 12)
Generating battery 13 (trial 13)
Generating battery 14 (trial 14)
Generating battery 15 (trial 15)
Generating battery 16 (trial 16)
Generating battery 17 (trial 17)
Generating battery 18 (trial 18)
Generating battery 19 (trial 19)
Generating battery 20 (trial 20)
Generating battery 21 (trial 21)
Generating battery 22 (trial 22)
Generating battery 23 (trial 23)
⚠️ DFN fail at t=255
❌ battery removed
Generating battery 23 (trial 24)
Generating battery 24 (trial 25)
Generating battery 25 (trial 26)
Generating battery 26 (trial 27)
Generating battery 27 (trial 28)
Generating battery 28 (t

KeyboardInterrupt: 

In [5]:
import pybamm
import numpy as np
import pandas as pd
import os

# ==========================
# 設定
# ==========================
N_SIM = 1000
TOTAL_TIME = 1000
SAVE_DIR = "dataset_lstm_NoRandom1000"
os.makedirs(SAVE_DIR, exist_ok=True)

options = {
    "SEI": "ec reaction limited",
    "lithium plating": "irreversible",
}

# ==========================
# 成功数管理
# ==========================
save_id = 0
trial_id = 0

# ==========================
# 成功するまで回す
# ==========================
while save_id < N_SIM:

    print(f"Generating battery {save_id} (trial {trial_id})")
    trial_id += 1

#     # ==========================
#     # 🎲 ランダム劣化
#     # ==========================
#     k_sei = 10**np.random.uniform(-16, -13)
#     k_plating = 10**np.random.uniform(-11, -8)
#     dead_li = 10**np.random.uniform(-7, -4)

#     eps_n = np.random.uniform(0.5, 0.7)
#     eps_p = np.random.uniform(0.4, 0.6)

#     r_n = np.random.uniform(5e-6, 15e-6)
#     r_p = np.random.uniform(5e-6, 15e-6)

#     T = np.random.uniform(273, 313)

#     base_param = pybamm.ParameterValues("OKane2022")
#     base_param.update({
#         "SEI kinetic rate constant [m.s-1]": k_sei,
#         "Lithium plating kinetic rate constant [m.s-1]": k_plating,
#         "Lithium plating transfer coefficient": 0.5,
#         "Dead lithium decay constant [s-1]": dead_li,
#         "Negative electrode active material volume fraction": eps_n,
#         "Positive electrode active material volume fraction": eps_p,
#         "Negative particle radius [m]": r_n,
#         "Positive particle radius [m]": r_p,
#         "Ambient temperature [K]": T,
#         "Initial temperature [K]": T,
#     })

    # ==========================
    # 固定劣化パラメータ
    # ==========================
    k_sei = 1e-14                # SEI反応速度
    k_plating = 1e-9             # Li析出反応速度
    dead_li = 1e-6               # デッドリチウム減衰定数

    eps_n = 0.6                  # 負極活物質体積分率
    eps_p = 0.5                  # 正極活物質体積分率

    r_n = 1e-5                   # 負極粒子半径 [m]
    r_p = 1e-5                   # 正極粒子半径 [m]

    T = 298                       # 温度 [K]

    base_param = pybamm.ParameterValues("OKane2022")
    base_param.update({
        "SEI kinetic rate constant [m.s-1]": k_sei,
        "Lithium plating kinetic rate constant [m.s-1]": k_plating,
        "Lithium plating transfer coefficient": 0.5,
        "Dead lithium decay constant [s-1]": dead_li,
        "Negative electrode active material volume fraction": eps_n,
        "Positive electrode active material volume fraction": eps_p,
        "Negative particle radius [m]": r_n,
        "Positive particle radius [m]": r_p,
        "Ambient temperature [K]": T,
        "Initial temperature [K]": T,
    })    
    
    # ==========================
    # 🎲 初期SOCランダム化
    # ==========================
#     v_init = np.random.uniform(3.5, 3.9)
    v_init = 3.7  # 例：すべて同じ初期電圧
    model_init = pybamm.lithium_ion.DFN(options=options)

    init_exp = pybamm.Experiment([
        f"Discharge at 0.2C until {v_init} V",
        "Rest for 5 hour"
    ])

    sim_init = pybamm.Simulation(
        model_init,
        parameter_values=base_param.copy(),
        experiment=init_exp
    )

    try:
        solution_init = sim_init.solve()
    except pybamm.SolverError:
        print("❌ init fail")
        continue

    # ==========================
    # Driveモデル
    # ==========================
    drive_param = base_param.copy()
    drive_param.update({
        "Current function [A]": pybamm.InputParameter("Current function [A]")
    })

    model = pybamm.lithium_ion.DFN(options=options)
    model.set_initial_conditions_from(solution_init)

    sim = pybamm.Simulation(model, parameter_values=drive_param)

    try:
        sim.solve([0, 1], inputs={"Current function [A]": 0})
    except pybamm.SolverError:
        print("❌ drive init fail")
        continue

    esoh_solver = pybamm.lithium_ion.ElectrodeSOHSolver(
        parameter_values=base_param.copy()
    )

    # ==========================
    # ログ
    # ==========================
    time_log = []
    fcc_log = []
    current_log = []
    voltage_log = []

    I_now = 0
    I_max = np.random.uniform(1, 4)
    max_step = np.random.uniform(0.01, 0.2)

    valid_battery = True

    # ==========================
    # 毎秒ループ
    # ==========================
    for t in range(TOTAL_TIME):

        solution = sim.solution
        V_now = solution["Terminal voltage [V]"].entries[-1]

        # 電圧制約
        if V_now > 4.19:
            I_now = min(I_now, 0)

        if V_now < 3.01:
            I_now = max(I_now, 0)

        # ランダム更新
        I_now += np.random.uniform(-max_step, max_step)
        I_now = np.clip(I_now, -I_max, I_max)

        # DFN step
        try:
            sim.step(dt=1, inputs={"Current function [A]": I_now})
        except pybamm.SolverError:
            print(f"⚠️ DFN fail at t={t}")
            valid_battery = False
            break

        solution = sim.solution

        Q_n = solution["Negative electrode capacity [A.h]"].entries[-1]
        Q_p = solution["Positive electrode capacity [A.h]"].entries[-1]
        Q_Li = solution["Total lithium capacity [A.h]"].entries[-1]

        # eSOH
        try:
            esoh_sol = esoh_solver.solve({
                "Q_n": Q_n,
                "Q_p": Q_p,
                "Q_Li": Q_Li
            })
            FCC = esoh_sol["Q"]
        except pybamm.SolverError:
            print(f"⚠️ eSOH fail at t={t}")
            valid_battery = False
            break

        V = solution["Terminal voltage [V]"].entries[-1]

        time_log.append(t)
        fcc_log.append(FCC)
        current_log.append(-I_now)
        voltage_log.append(V)

    # ==========================
    # 保存（成功セルのみ）
    # ==========================
    if valid_battery and len(time_log) == TOTAL_TIME:

        df = pd.DataFrame({
            "time_s": time_log,
            "current_A": current_log,
            "voltage_V": voltage_log,
            "FCC_Ah": fcc_log
        })

        df.to_csv(f"{SAVE_DIR}/battery_{save_id:04d}.csv", index=False)
        save_id += 1

    else:
        print("❌ battery removed")

print("All datasets generated successfully.")

Generating battery 0 (trial 0)
Generating battery 1 (trial 1)
Generating battery 2 (trial 2)
Generating battery 3 (trial 3)
Generating battery 4 (trial 4)
Generating battery 5 (trial 5)
Generating battery 6 (trial 6)
Generating battery 7 (trial 7)
Generating battery 8 (trial 8)
Generating battery 9 (trial 9)
Generating battery 10 (trial 10)
Generating battery 11 (trial 11)
Generating battery 12 (trial 12)
Generating battery 13 (trial 13)
Generating battery 14 (trial 14)
Generating battery 15 (trial 15)
Generating battery 16 (trial 16)
Generating battery 17 (trial 17)
Generating battery 18 (trial 18)
Generating battery 19 (trial 19)
Generating battery 20 (trial 20)
Generating battery 21 (trial 21)
Generating battery 22 (trial 22)
Generating battery 23 (trial 23)
Generating battery 24 (trial 24)
Generating battery 25 (trial 25)
Generating battery 26 (trial 26)
Generating battery 27 (trial 27)
Generating battery 28 (trial 28)
Generating battery 29 (trial 29)
Generating battery 30 (trial 3

In [6]:
import os
import glob
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import matplotlib.pyplot as plt

# ==========================
# 設定
# ==========================
SEQ_LEN = 50   # 時間窓
BATCH_SIZE = 64
EPOCHS = 50
LR = 1e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# TRAIN_DIR = r"D:\BMS\FCC_PyBaMMDara_NoRandom\Train"
# VAL_DIR   = r"D:\BMS\FCC_PyBaMMDara_NoRandom\Val"
# TEST_DIR  = r"D:\BMS\FCC_PyBaMMDara_NoRandom\Test"

TRAIN_DIR = r"D:\Yamato\dataset_lstm_NoRandom1000\Train"
VAL_DIR   = r"D:\Yamato\dataset_lstm_NoRandom1000\Val"
TEST_DIR  = r"D:\Yamato\dataset_lstm_NoRandom1000\Test"

# ==========================
# Train全体で標準化用統計量を計算
# ==========================
train_files = glob.glob(os.path.join(TRAIN_DIR, "*.csv"))
all_train_data = np.concatenate([pd.read_csv(f)[["current_A","voltage_V"]].values for f in train_files])
MEAN = all_train_data.mean(axis=0)
STD  = all_train_data.std(axis=0)

# ==========================
# Datasetクラス
# ==========================
class BatteryDataset(Dataset):
    def __init__(self, folder_path, seq_len=SEQ_LEN, mean=MEAN, std=STD):
        self.seq_len = seq_len
        self.samples = []
        self.mean = mean
        self.std = std

        files = glob.glob(os.path.join(folder_path, "*.csv"))
        for f in files:
            df = pd.read_csv(f)
            data = df[["current_A", "voltage_V"]].values
            # 全体標準化
            data = (data - self.mean) / self.std
            # ΔFCCをターゲットにする
            fcc = df["FCC_Ah"].values
            delta_fcc = np.diff(fcc, prepend=fcc[0])  # ΔFCC[0] = 0
            for i in range(len(df) - seq_len):
                x = data[i:i+seq_len]
                y = delta_fcc[i+seq_len-1]
                self.samples.append((x, y))
            # 元のFCC保持（単一バッテリー用）も可能
            self.fcc = fcc

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        x, y = self.samples[idx]
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)

# ==========================
# LSTMモデル
# ==========================
class FCC_LSTM(nn.Module):
    def __init__(self, input_size=2, hidden_size=64, num_layers=2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=0.2)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]  # 最後の時刻
        out = self.fc(out)
        return out.squeeze()

# ==========================
# データロード
# ==========================
train_dataset = BatteryDataset(TRAIN_DIR)
val_dataset   = BatteryDataset(VAL_DIR)
test_dataset  = BatteryDataset(TEST_DIR)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE)

# ==========================
# モデル・最適化
# ==========================
model = FCC_LSTM().to(DEVICE)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

# ==========================
# 学習ループ
# ==========================
for epoch in range(EPOCHS):
    model.train()
    train_loss = 0
    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        pred = model(x)
        loss = criterion(pred, y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)

    model.eval()
    val_loss = 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            pred = model(x)
            val_loss += criterion(pred, y).item()
    val_loss /= len(val_loader)

    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.6e} | Val Loss: {val_loss:.6e}")

# ==========================
# テスト時は ΔFCC を累積して元のFCCを再構築
# ==========================
def predict_battery(file_path):
    df = pd.read_csv(file_path)
    data = df[["current_A","voltage_V"]].values
    data = (data - MEAN)/STD
    fcc_true = df["FCC_Ah"].values

    dataset = BatteryDataset(folder_path=os.path.dirname(file_path))
    # ここでは単一ファイル用
    x_seq = []
    for i in range(len(df)-SEQ_LEN):
        x_seq.append(data[i:i+SEQ_LEN])
    x_seq = torch.tensor(x_seq, dtype=torch.float32).to(DEVICE)

    model.eval()
    with torch.no_grad():
        delta_pred = model(x_seq).cpu().numpy()
    # ΔFCC を累積して元のFCCに変換
    fcc_pred = np.cumsum(delta_pred) + fcc_true[0]

    # プロット
    plt.figure(figsize=(12,5))
    plt.plot(fcc_true[SEQ_LEN-1:], label="True FCC", alpha=0.7)
    plt.plot(fcc_pred, '--', label="Predicted FCC", alpha=0.7)
    plt.xlabel("Time step")
    plt.ylabel("FCC [Ah]")
    plt.title(f"FCC Prediction vs True ({os.path.basename(file_path)})")
    plt.legend()
    plt.show()

# ==========================
# サンプル予測
# ==========================
predict_battery(r"D:\BMS\FCC_PyBaMMDara_NoRandom\Test\battery_0804.csv")

Epoch 1/50 | Train Loss: 8.203442e-06 | Val Loss: 3.461451e-08
Epoch 2/50 | Train Loss: 6.628903e-09 | Val Loss: 1.492354e-10
Epoch 3/50 | Train Loss: 2.343976e-09 | Val Loss: 4.132906e-10
Epoch 4/50 | Train Loss: 1.811576e-09 | Val Loss: 2.332468e-09
Epoch 5/50 | Train Loss: 1.593200e-09 | Val Loss: 2.746931e-09
Epoch 6/50 | Train Loss: 1.478911e-09 | Val Loss: 8.488493e-09
Epoch 7/50 | Train Loss: 1.391660e-09 | Val Loss: 2.872651e-10
Epoch 8/50 | Train Loss: 1.319806e-09 | Val Loss: 7.693318e-11
Epoch 9/50 | Train Loss: 1.249269e-09 | Val Loss: 2.007074e-11
Epoch 10/50 | Train Loss: 1.202879e-09 | Val Loss: 1.065860e-11
Epoch 11/50 | Train Loss: 1.164085e-09 | Val Loss: 1.892113e-10
Epoch 12/50 | Train Loss: 1.128099e-09 | Val Loss: 8.429567e-10
Epoch 13/50 | Train Loss: 1.083767e-09 | Val Loss: 4.549823e-11
Epoch 14/50 | Train Loss: 1.059400e-09 | Val Loss: 8.487365e-11
Epoch 15/50 | Train Loss: 1.026748e-09 | Val Loss: 6.925592e-10
Epoch 16/50 | Train Loss: 1.002395e-09 | Val Loss

FileNotFoundError: [Errno 2] No such file or directory: 'D:\\BMS\\FCC_PyBaMMDara_NoRandom\\Test\\battery_0804.csv'